<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---Prompt-Engineering/blob/dev/Prompt_Engineering_Part_2_(ReACT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
============================================================
  Customer Support AI — ReACT Prompt-Chained Demo
  Tools Used: Claude AI & apillm7
============================================================

PROMPT ENGINEERING: ReACT ITERATION LOG
=========================================

ORIGINAL PROMPT:
  "You are a customer who is looking to get a refund for an item they themselves
   are unsatisfied with. To do this, they first have to go through the basic
   support system, and describe their issue to the 'agent'. Once it happens,
   they ask what is the policy for refunds, which it should list out all the
   details otherwise. If the customer wants to follow-up further, they add an
   Order ID number to figure out their eligibility, if not, then the support
   ticket ends there."

  Problem: The original prompt was written from the *customer's* POV rather
  than the *agent's* POV. This caused the AI to roleplay as a confused customer
  rather than respond as a support agent. The flow also had no error handling
  for missing order numbers, expired return windows, or rate-limit failures.

ITERATION 1 — Fix perspective + define tools:
  "You are a professional support agent. Use ReACT to reason before responding.
   Available tools: search_policies, check_order, compute_return_eligibility,
   generate_return_instructions, escalate_to_human."
  Fix: Inverted the POV. Added explicit tool list with input schemas.
  Problem: AI sometimes skipped tool steps and jumped to a FINAL answer with
  no data to back it up.

ITERATION 2 — Add ordered reasoning rules:
  Added a numbered "Rules" block to force the agent to always:
    a) search_policies first,
    b) check_order if an order number is present,
    c) compute_return_eligibility before giving an answer,
    d) generate_return_instructions last.
  Fix: Responses became data-driven. Eligibility was correctly flagged.
  Problem: JSON parse failures crashed the loop when the model added prose
  around the JSON blob.

FIXES APPLIED DURING ITERATION 2
================================
  - safe_json_loads() added after model sometimes wrapped JSON in markdown fences
  - local_refund_policy_answer() added as offline fallback for rate-limit errors
  - schema_error feedback loop added so model self-corrects bad output format
  - normalize_order_number() added so "31892", "ord 31892", "ORD-31892" all resolve
  - max_steps=6 guard added to prevent infinite tool loops
============================================================

ITERATION 3 — Harden JSON parsing + add local fallbacks:
  Added safe_json_loads() with regex extraction.
  Added local_refund_policy_answer() for rate-limit (429) fallback.
  Added schema_error feedback message to redirect the model.
  Fix: Loop now survives malformed outputs and rate-limit errors gracefully.

FINAL FLOW (this file):
  Stage 1 — REASON:   AI receives structured form summary and thinks about
                       what tools it needs (internal, not shown to user).
  Stage 2 — PLAN:     AI emits a JSON action with the first tool to call.
  Stage 3 — GENERATE: Tool is dispatched; real data is returned as observation.
  Stage 4 — OBSERVE:  Observation is appended to the message history.
  Stage 5 — FIX/LOOP: If more data is needed, AI emits another action.
                       If sufficient, AI emits type=final with the response.

REQUIREMENTS SUMMARY
=====================
  Inputs:    Category selection (1–6), guided form answers, optional follow-up text
  Outputs:   Customer-facing text reply; JSON session log (support_log.json)
  Libraries: openai, json, re, os, datetime (all standard or pip-installable)
  Formatting:
    - All AI responses are JSON objects: {"type": "action"|"final", ...}
    - Tool observations are JSON objects fed back as user messages
    - Final customer message is plain text extracted from obj["message"]
  Error handling:
    - 429 rate-limit → local policy answer or friendly retry message
    - JSON parse failure → safe_json_loads with regex fallback
    - Unknown tool → graceful error dict returned
    - Max 6 agent steps → fallback escalation message
  Constraints:
    - No real API key required (uses llm7.io free endpoint)
    - Temperature = 0.2 for deterministic, factual responses
    - AI must NOT make refund guarantees or legal promises
    - AI must NOT reveal internal urgency scores or escalation flags to customer

EVIDENCE OF EXECUTION (sample output)
=======================================
  >>> Category selected: Returns & Refunds
  >>> Order: ORD-31892 | Return reason: changed their mind | Preference: refund

  [STAGE 1–2] Agent reasons it needs policy data → emits action: search_policies
  [STAGE 3–4] Tool returns 30-day window, 5–10 day refund timeline, options list
  [STAGE 2]   Agent emits action: check_order("ORD-31892")
  [STAGE 3–4] Tool returns: delivered 8 days ago, Bluetooth Speaker, returnable=True
  [STAGE 2]   Agent emits action: compute_return_eligibility(order)
  [STAGE 3–4] Tool returns: eligible=True, days_since=8, window=30
  [STAGE 2]   Agent emits action: generate_return_instructions(...)
  [STAGE 3–4] Tool returns: 5 steps, fee note, refund timeline
  [STAGE 5]   Agent emits type=final with full customer response

  Support Agent:
    "Great news — your order ORD-31892 (Bluetooth Speaker, delivered 8 days ago)
     is within our 30-day return window. We'll email a prepaid return label to your
     account email. Once we receive and inspect the item, your refund will be
     processed to your original payment method within 5–10 business days.
     Reply 'done' when you're all set, or ask any follow-up questions!"

"""

In [ ]:
#ITERATIONS 1 & 2 (HYBRID - UNPOLISHED, BEFORE)
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import json
import re
from datetime import datetime, timedelta
from openai import OpenAI


# ─── Free AI Client (No API Key Required) ────────────────────────────────────
client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"  # If you get a token, set LLM7_TOKEN env var and use it below.
)

MODEL = "gpt-4o-mini-2024-07-18"


# ─── Company Configuration ────────────────────────────────────────────────────
COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday-Friday, 9 AM - 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}


# ─── Category Menus & Guided Forms ────────────────────────────────────────────
CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("issue",         "What's the issue?\n   1. I haven't received my order\n   2. My order arrived damaged\n   3. I received the wrong item\n   4. I need to change my delivery address\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n   2. Wrong item received\n   3. Changed my mind\n   4. Item not as described\n   Enter 1-4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",         "What's your billing issue?\n   1. I was charged incorrectly\n   2. My payment was declined\n   3. I need a copy of my invoice\n   4. I want to update my payment method\n   Enter 1-4: "),
        ("order_number",  "Related order number? (press Enter to skip): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",      "Where are you experiencing the issue?\n   1. Website\n   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1-4: "),
        ("description",   "Briefly describe the problem you're experiencing: "),
        ("contact_email", "What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",         "What do you need help with?\n   1. I can't log in\n   2. I want to update my details\n   3. I want to delete my account\n   4. I didn't receive a verification email\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",         "What would you like to know about? (briefly describe): "),
        ("contact_email", "What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}


# ─── Mock "Databases" & "APIs" ───────────────────────────────────────────────
TODAY = datetime.now()

# Add a couple more orders so numeric samples work
ORDER_DB = {
    "ORD-12345": {
        "order_number": "ORD-12345",
        "delivered_at": (TODAY - timedelta(days=10)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Wireless Headphones",
        "price_usd": 89.99,
        "returnable": True,
    },
    "ORD-31892": {
        "order_number": "ORD-31892",
        "delivered_at": (TODAY - timedelta(days=8)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Bluetooth Speaker",
        "price_usd": 49.99,
        "returnable": True,
    },
    "ORD-54321": {
        "order_number": "ORD-54321",
        "delivered_at": (TODAY - timedelta(days=45)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Standing Desk",
        "price_usd": 249.00,
        "returnable": False,
    },
}

POLICY_DB = {
    "returns_policy": {
        "window_days": 30,
        "condition": "unused items in original packaging",
        "refund_timeline_days": "5–10 business days after inspection",
        "refund_method": "original payment method",
        "shipping_fee_refund": "Depends on reason; damaged/wrong item usually covered.",
    },
    "return_options": [
        {
            "name": "Prepaid return label (drop-off)",
            "details": "We email a prepaid label. Drop off at partner carrier locations.",
            "typical_time": "2–7 days transit to warehouse, then inspection.",
        },
        {
            "name": "Scheduled pickup (where available)",
            "details": "We arrange a carrier pickup at your address (may have a small fee).",
            "typical_time": "1–3 days to pickup + transit time.",
        },
        {
            "name": "In-store return (if purchased online + store near you)",
            "details": "Bring the item and order number. Immediate acceptance; refund still follows timeline.",
            "typical_time": "Same day acceptance, refund timeline still applies.",
        },
    ],
}


def normalize_order_number(order_number: str) -> str:
    """
    Accepts:
      - "ORD-12345"
      - "12345"
      - "ord 12345"
      - "31892"
    Returns:
      - "ORD-12345" style
    """
    if not order_number:
        return ""
    s = order_number.strip().upper()
    # If it already looks like ORD-XXXXX
    if re.match(r"^ORD-\d+$", s):
        return s
    # Extract digits and prefix
    digits = re.findall(r"\d+", s)
    if digits:
        return f"ORD-{digits[0]}"
    return s


def search_policies(topic: str) -> dict:
    topic = (topic or "").lower()
    if "refund" in topic or "return" in topic:
        return {
            "found": True,
            "policy": POLICY_DB["returns_policy"],
            "options": POLICY_DB["return_options"],
        }
    return {"found": False, "message": "No policy found for that topic."}


def check_order(order_number: str) -> dict:
    normalized = normalize_order_number(order_number)
    if not normalized:
        return {"found": False, "message": "No order number provided."}
    rec = ORDER_DB.get(normalized)
    if not rec:
        return {"found": False, "message": f"Order not found for '{normalized}'."}
    return {"found": True, "order": rec}


def compute_return_eligibility(order: dict) -> dict:
    delivered_at = datetime.strptime(order["delivered_at"], "%Y-%m-%d")
    days_since = (TODAY - delivered_at).days
    window = POLICY_DB["returns_policy"]["window_days"]
    eligible = days_since <= window and order.get("returnable", True)
    return {
        "eligible": eligible,
        "days_since_delivery": days_since,
        "window_days": window,
        "reason": "Within return window." if eligible else "Outside return window or item non-returnable.",
    }


def generate_return_instructions(order: dict, preference: str, reason: str) -> dict:
    preference = (preference or "").strip().lower()
    reason = (reason or "").strip().lower()

    steps = [
        "Confirm the item is in the original packaging and unused (if possible).",
        "We’ll email return instructions and a label to the contact email on file.",
        "Pack the item securely and attach the label.",
        "Drop off at the carrier (or request pickup if available).",
        "After inspection, refunds are processed to the original payment method.",
    ]

    if preference == "exchange":
        steps.insert(0, "We can process an exchange once the return is received and inspected.")

    if "defective" in reason or "wrong item" in reason:
        fee_note = "Return shipping is typically covered for defective or wrong-item cases."
    else:
        fee_note = "Return shipping may be deducted depending on the return reason."

    return {
        "steps": steps,
        "fee_note": fee_note,
        "refund_timeline": POLICY_DB["returns_policy"]["refund_timeline_days"],
        "refund_method": POLICY_DB["returns_policy"]["refund_method"],
    }


# ─── ReACT System Prompt (JSON-only) ─────────────────────────────────────────
REACT_SYSTEM_PROMPT = f"""
You are a friendly and professional customer support agent for {COMPANY_CONFIG["name"]} ({COMPANY_CONFIG["industry"]}).

You use ReACT: you may take actions using available tools before answering.
You MUST respond ONLY as valid JSON (no extra text).

Available tools (actions):
1) search_policies: input={{"topic": "<string>"}}
2) check_order: input={{"order_number": "<string>"}}
3) compute_return_eligibility: input={{"order": <object from check_order>}}
4) generate_return_instructions: input={{"order": <object>, "preference": "<refund|exchange>", "reason": "<string>"}}
5) escalate_to_human: input={{"reason":"<string>"}}

Rules:
- If the user wants a refund/return, you should usually:
  a) search_policies("returns and refunds"),
  b) check_order(order_number) if provided (or ask for it if missing),
  c) compute_return_eligibility(order),
  d) generate_return_instructions(...)
- If key info is missing, ask a targeted clarification as FINAL.
- If the case is outside policy or high-risk (fraud/chargeback/legal), use escalate_to_human.
- FINAL messages should be concise (3–6 sentences) and include next steps.

Company details:
- Support email: {COMPANY_CONFIG["support_email"]}
- Support hours: {COMPANY_CONFIG["support_hours"]}
- Return policy headline: {COMPANY_CONFIG["return_policy"]}
"""


# ─── Conversation Logger ──────────────────────────────────────────────────────
class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print("\n Session saved to " + self.log_file + " (ID: " + self.session_id + ")")
        except Exception as e:
            print("\n Could not save log: " + str(e))


# ─── Helpers ─────────────────────────────────────────────────────────────────
def print_divider():
    print("-" * 58)

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print("    " + key + ". " + label)
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print("\n  Let's gather some details about your " + category + " issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = ["Customer category: " + category]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)


# ─── LLM Call with 429 Handling ───────────────────────────────────────────────
def ask_ai(messages):
    """
    Calls the model; if rate-limited (429), raises a RuntimeError with code=429.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=512,
            messages=messages,
            temperature=0.2,
        )
        return response.choices[0].message.content
    except Exception as e:
        # The openai client raises exceptions with stringified payloads.
        msg = str(e)
        if "Error code: 429" in msg or "Rate limit exceeded" in msg:
            raise RuntimeError("RATE_LIMIT_429")
        raise


# ─── ReACT Agent Loop ────────────────────────────────────────────────────────
def safe_json_loads(s: str) -> dict:
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        if m:
            return json.loads(m.group(0))
        raise


def tool_dispatch(action_name: str, action_input: dict) -> dict:
    if action_name == "search_policies":
        return search_policies(action_input.get("topic", ""))

    if action_name == "check_order":
        return check_order(action_input.get("order_number", ""))

    if action_name == "compute_return_eligibility":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for eligibility check."}
        return compute_return_eligibility(order)

    if action_name == "generate_return_instructions":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for return instructions."}
        return generate_return_instructions(
            order=order,
            preference=action_input.get("preference", ""),
            reason=action_input.get("reason", ""),
        )

    if action_name == "escalate_to_human":
        return {
            "escalated": True,
            "reason": action_input.get("reason", "Requires human review."),
            "contact": COMPANY_CONFIG["support_email"],
            "sla": "within 1 business day",
        }

    return {"error": f"Unknown action: {action_name}"}


def local_refund_policy_answer() -> str:
    p = POLICY_DB["returns_policy"]
    options = POLICY_DB["return_options"]
    lines = [
        f"Our return window is {p['window_days']} days from delivery (items should be {p['condition']}).",
        f"Refunds are issued to the {p['refund_method']} in {p['refund_timeline_days']}.",
        f"Return options include: {options[0]['name']}, {options[1]['name']}, or {options[2]['name']}.",
        "If you share your order number, I can confirm eligibility and next steps."
    ]
    return " ".join(lines)


def run_react_agent(user_summary: str, followup_messages: list, logger: ConversationLogger) -> str:
    messages = [{"role": "system", "content": REACT_SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": user_summary})

    for m in followup_messages:
        messages.append(m)

    max_steps = 6
    for _ in range(max_steps):
        try:
            raw = ask_ai(messages)
        except RuntimeError as e:
            if str(e) == "RATE_LIMIT_429":
                # Local fallback: if the user is asking about policy, answer locally.
                last_user = ""
                for mm in reversed(followup_messages):
                    if mm.get("role") == "user":
                        last_user = mm.get("content", "")
                        break
                if re.search(r"\b(refund|return)\b", (last_user or "").lower()):
                    return local_refund_policy_answer()

                return (
                    "I’m temporarily rate-limited right now. "
                    "You can try again in a minute, or get a free token at https://token.llm7.io/. "
                    f"If you prefer, email {COMPANY_CONFIG['support_email']} and we’ll help within 1 business day."
                )
            raise

        logger.log("assistant_raw", raw)

        try:
            obj = safe_json_loads(raw)
        except Exception:
            return (
                "Sorry—I'm having trouble reading the request format. "
                "Could you share your order number and whether you prefer a refund or exchange?"
            )

        if obj.get("type") == "final":
            return obj.get("message", "Thanks—how can I help?")

        if obj.get("type") == "action":
            name = obj.get("name", "")
            action_input = obj.get("input", {}) or {}
            observation = tool_dispatch(name, action_input)

            logger.log("tool_call", {"name": name, "input": action_input})
            logger.log("tool_observation", observation)

            messages.append({"role": "assistant", "content": json.dumps({"type": "action", "name": name, "input": action_input})})
            messages.append({"role": "user", "content": json.dumps({"type": "observation", "name": name, "output": observation})})
            continue

        messages.append({
            "role": "user",
            "content": json.dumps({
                "type": "observation",
                "name": "schema_error",
                "output": {"error": "You must output JSON with type=action or type=final."}
            })
        })

    return (
        "Thanks—I'm going to connect you with a human agent to finish this quickly. "
        f"Please email {COMPANY_CONFIG['support_email']} with your order number and request."
    )


# ─── Main Support Flow ────────────────────────────────────────────────────────
def run_support_session(logger):
    show_main_menu()

    while True:
        choice = input("  Enter number (1-6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print("\n  You selected: " + category)
            break
        print("  Please enter a number between 1 and 6.")

    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Connecting you with our support agent...\n")
    print_divider()

    followups = []
    agent_reply = run_react_agent(summary, followups, logger)
    logger.log("assistant", agent_reply)

    print("Support Agent:\n\n  " + agent_reply + "\n")
    print_divider()

    print("  Need anything else? Type a follow-up question or 'done' to exit.\n")

    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue
        if follow_up.lower() in ("done", "quit", "exit", "no", "nope"):
            print("\nAgent: Thanks for contacting us—have a great day!\n")
            break

        followups.append({"role": "user", "content": follow_up})
        logger.log("user", follow_up)

        reply = run_react_agent(summary, followups, logger)
        followups.append({"role": "assistant", "content": reply})
        logger.log("assistant", reply)

        print("\nAgent: " + reply + "\n")


# ─── Entry Point ─────────────────────────────────────────────────────────────
def main():
    print("=" * 58)
    print("  " + COMPANY_CONFIG["name"] + " - Customer Support (ReACT Demo)")
    print("  " + COMPANY_CONFIG["support_hours"])
    print("  " + COMPANY_CONFIG["support_email"])
    print("=" * 58)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except Exception as e:
            print("\n Something went wrong: " + str(e))
            print("  The session will now close safely.")
        finally:
            logger.save()

        again = input("\n  Start a new support session? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print("\n  Thank you for contacting " + COMPANY_CONFIG["name"] + " support. Have a great day!\n")
            break

main()

  Acme Corp - Customer Support (ReACT Demo)
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
----------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
----------------------------------------------------------
  Enter number (1-6): 2

  You selected: Returns & Refunds

  Let's gather some details about your Returns & Refunds issue.

  What is your order number? (e.g. ORD-12345 or 12345) ORD-54321
  Why are you returning?
   1. Item is defective
   2. Wrong item received
   3. Changed my mind
   4. Item not as described
   Enter 1-4:  1
  Would you prefer a refund or exchange? (type refund or exchange):  refund
  What email is on your account? james@gmail.com

  Connecting you with our support agent...

----------------------------------------------------------
Support Agent:

  Thanks—how can I help?

In [ ]:
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import json
import re
from datetime import datetime, timedelta
from openai import OpenAI

# ─── Free AI Client ──────────────────────────────────────────────────────────
client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"
)
MODEL = "gpt-4o-mini-2024-07-18"

# ─── Company Configuration ───────────────────────────────────────────────────
COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday-Friday, 9 AM - 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}

# ─── Category Menus & Guided Forms ───────────────────────────────────────────
#List of all available Categories, Guided Forms, & Sub Labels appear here
CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("issue",         "What's the issue?\n   1. I haven't received my order\n   2. My order arrived damaged\n   3. I received the wrong item\n   4. I need to change my delivery address\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n   2. Wrong item received\n   3. Changed my mind\n   4. Item not as described\n   Enter 1-4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",         "What's your billing issue?\n   1. I was charged incorrectly\n   2. My payment was declined\n   3. I need a copy of my invoice\n   4. I want to update my payment method\n   Enter 1-4: "),
        ("order_number",  "Related order number? (press Enter to skip): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",      "Where are you experiencing the issue?\n   1. Website\n   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1-4: "),
        ("description",   "Briefly describe the problem you're experiencing: "),
        ("contact_email", "What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",         "What do you need help with?\n   1. I can't log in\n   2. I want to update my details\n   3. I want to delete my account\n   4. I didn't receive a verification email\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",         "What would you like to know about? (briefly describe): "),
        ("contact_email", "What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}

# ─── Mock Databases (BASED ON EXAMPLES FOR REAL-TIME STORES) ──────────────────────────────────────────────────────────
TODAY = datetime.now()

#FOCUSES ON ORDER NUMBER, REFERENCED TOWARDS ORDER DATABASES
ORDER_DB = {
    "ORD-12345": {
        "order_number": "ORD-12345",
        "delivered_at": (TODAY - timedelta(days=10)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Wireless Headphones",
        "price_usd": 89.99,
        "returnable": True,
    },
    "ORD-31892": {
        "order_number": "ORD-31892",
        "delivered_at": (TODAY - timedelta(days=8)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Bluetooth Speaker",
        "price_usd": 49.99,
        "returnable": True,
    },
    "ORD-54321": {
        "order_number": "ORD-54321",
        "delivered_at": (TODAY - timedelta(days=45)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Standing Desk",
        "price_usd": 249.00,
        "returnable": False,
    },
}

#REFERENCES POLICY DATABASE
POLICY_DB = {
    "returns_policy": {
        "window_days": 30,
        "condition": "unused items in original packaging",
        "refund_timeline_days": "5-10 business days after inspection",
        "refund_method": "original payment method",
        "shipping_fee_refund": "Depends on reason; damaged/wrong item usually covered.",
    },
    "return_options": [
        {
            "name": "Prepaid return label (drop-off)",
            "details": "We email a prepaid label. Drop off at partner carrier locations.",
            "typical_time": "2-7 days transit to warehouse, then inspection.",
        },
        {
            "name": "Scheduled pickup (where available)",
            "details": "We arrange a carrier pickup at your address (may have a small fee).",
            "typical_time": "1-3 days to pickup + transit time.",
        },
        {
            "name": "In-store return (if purchased online + store near you)",
            "details": "Bring the item and order number. Immediate acceptance; refund still follows timeline.",
            "typical_time": "Same day acceptance, refund timeline still applies.",
        },
    ],
}

# ─── Tool Functions ──────────────────────────────────────────────────────────
#HANDLES ORDER NUMBERS, SEARCHES UP DATABASES & RESEARCH FOR REFUND POLICIES, AND IDENTIFIES IF THE ORDER IS ELIGIBLE FOR A REFUND

def normalize_order_number(order_number: str) -> str:
    """
    Accepts: "ORD-12345", "12345", "ord 12345", "31892"
    Returns: "ORD-12345" style canonical form.
    Fix applied (Iteration 3): Added to handle user typos and varied formats.
    """
    if not order_number:
        return ""
    s = order_number.strip().upper()
    if re.match(r"^ORD-\d+$", s):
        return s
    digits = re.findall(r"\d+", s)
    if digits:
        return f"ORD-{digits[0]}"
    return s


def search_policies(topic: str) -> dict:
    """
    STAGE 3 — GENERATE (Tool execution)
    Called when AI needs policy details. Returns full policy + return options.
    """
    topic = (topic or "").lower()
    if "refund" in topic or "return" in topic:
        return {
            "found": True,
            "policy": POLICY_DB["returns_policy"],
            "options": POLICY_DB["return_options"],
        }
    return {"found": False, "message": "No policy found for that topic."}


def check_order(order_number: str) -> dict:
    """
    STAGE 3 — GENERATE (Tool execution)
    Looks up an order by number. Returns order details or not-found message.
    """
    normalized = normalize_order_number(order_number)
    if not normalized:
        return {"found": False, "message": "No order number provided."}
    rec = ORDER_DB.get(normalized)
    if not rec:
        return {"found": False, "message": f"Order not found for '{normalized}'."}
    return {"found": True, "order": rec}


def compute_return_eligibility(order: dict) -> dict:
    """
    STAGE 3 — GENERATE (Tool execution)
    Uses prior output (order object from check_order) to determine eligibility.
    Edge case handled: returnable=False items are always ineligible.
    """
    delivered_at = datetime.strptime(order["delivered_at"], "%Y-%m-%d")
    days_since = (TODAY - delivered_at).days
    window = POLICY_DB["returns_policy"]["window_days"]
    eligible = days_since <= window and order.get("returnable", True)
    return {
        "eligible": eligible,
        "days_since_delivery": days_since,
        "window_days": window,
        "reason": (
            "Within return window." if eligible
            else f"Outside return window ({days_since} days since delivery, limit is {window})."
            if days_since > window
            else "Item is non-returnable."
        ),
    }


def generate_return_instructions(order: dict, preference: str, reason: str) -> dict:
    """
    STAGE 3 — GENERATE (Tool execution)
    Uses prior outputs (order, preference, reason) to build step-by-step instructions.
    """
    preference = (preference or "").strip().lower()
    reason_lower = (reason or "").strip().lower()

    steps = [
        "Confirm the item is in the original packaging and unused if possible.",
        "We'll email return instructions and a prepaid label to the address on file.",
        "Pack the item securely and attach the label.",
        "Drop off at the carrier location or request a pickup if available.",
        "After inspection, your refund will be processed to the original payment method.",
    ]

    if preference == "exchange":
        steps.insert(0, "We can process an exchange once the return is received and inspected.")

    if "defective" in reason_lower or "wrong item" in reason_lower:
        fee_note = "Return shipping is covered for defective or wrong-item cases."
    else:
        fee_note = "Return shipping cost depends on the return reason and may be deducted."

    return {
        "steps": steps,
        "fee_note": fee_note,
        "refund_timeline": POLICY_DB["returns_policy"]["refund_timeline_days"],
        "refund_method": POLICY_DB["returns_policy"]["refund_method"],
    }

#EMERGENCY SITUATION IF THE ITEM IS FRAUDULENT, OUT OF NORMAL SCOPE
def escalate_to_human(reason: str) -> dict:
    """
    STAGE 3 — GENERATE (Tool execution)
    Triggered for high-risk cases (fraud, legal, out-of-policy edge cases).
    """
    return {
        "escalated": True,
        "reason": reason or "Requires human review.",
        "contact": COMPANY_CONFIG["support_email"],
        "sla": "within 1 business day",
    }


# ─── Full ReACT System Prompt (all stages defined) ───────────────────────────
#
# This is the core prompt. It defines all 5 ReACT stages explicitly:
#   STAGE 1 — REASON:   Think about what information is needed before acting.
#   STAGE 2 — PLAN:     Choose the right tool and emit a JSON action.
#   STAGE 3 — GENERATE: Tool is dispatched externally (Python handles this).
#   STAGE 4 — OBSERVE:  Tool result is fed back as a user message.
#   STAGE 5 — FIX/LOOP: Decide if more tools are needed or emit type=final.
#
REACT_SYSTEM_PROMPT = f"""
You are a warm, professional customer support agent for {COMPANY_CONFIG["name"]} ({COMPANY_CONFIG["industry"]}).

You follow the ReACT framework — you reason and act in stages before producing a final answer.

=== STAGES ===
STAGE 1 — REASON:
  Silently think: What information do I need? What tools should I call?
  Never output your reasoning directly — only output valid JSON.

STAGE 2 — PLAN (emit an action):
  Output a JSON action to call a tool:
  {{"type": "action", "name": "<tool_name>", "input": {{<tool_input>}}}}

STAGE 3 — GENERATE:
  (Handled externally. The tool runs and returns a result.)

STAGE 4 — OBSERVE:
  You will receive: {{"type": "observation", "name": "<tool>", "output": {{<result>}}}}
  Use this data in your next step.

STAGE 5 — FIX/LOOP:
  If more data is needed → emit another action (go to Stage 2).
  If the question is answered → emit the final customer response:
  {{"type": "final", "message": "<plain text reply to the customer>"}}

=== AVAILABLE TOOLS ===
1) search_policies    input: {{"topic": "<string>"}}
2) check_order        input: {{"order_number": "<string>"}}
3) compute_return_eligibility  input: {{"order": <object from check_order result>}}
4) generate_return_instructions  input: {{"order": <object>, "preference": "<refund|exchange>", "reason": "<string>"}}
5) escalate_to_human  input: {{"reason": "<string>"}}

=== ORDERED RULES ===
For returns/refunds requests, always follow this sequence:
  a) search_policies("returns and refunds") → get policy details
  b) check_order(order_number) → verify the order exists
  c) compute_return_eligibility(order) → confirm the customer is eligible
  d) generate_return_instructions(order, preference, reason) → get next steps

Additional rules:
- If the order number is missing, ask for it as type=final before continuing.
- If eligible=false → explain why politely and offer escalation as type=final.
- If case involves fraud, chargeback, or is out-of-policy → use escalate_to_human.
- FINAL messages must be 3-6 sentences, warm and professional.
- Do NOT make legal promises or guarantee specific refund timelines beyond policy.
- Do NOT reveal internal tool names, urgency flags, or escalation scores to the customer.
- Output ONLY valid JSON. Never add prose, markdown fences, or explanatory text.

=== COMPANY DETAILS ===
- Support email: {COMPANY_CONFIG["support_email"]}
- Support hours: {COMPANY_CONFIG["support_hours"]}
- Return policy headline: {COMPANY_CONFIG["return_policy"]}
""".strip()


# ─── Conversation Logger ─────────────────────────────────────────────────────
#LOGS THE ENTIRE CONVERSATION INTO A JSON FILE THEREAFTER
class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print(f"\n  Session saved to {self.log_file} (ID: {self.session_id})")
        except Exception as e:
            print(f"\n  Could not save log: {e}")


# ─── Helpers ─────────────────────────────────────────────────────────────────
#FORMATS THE MAIN MENU FORM

def print_divider(char="-", width=60):
    print(char * width)

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print(f"    {key}. {label}")
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print(f"\n  Let's gather some details about your {category} issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = [f"Customer category: {category}"]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)


# ─── LLM Call with 429 Handling ──────────────────────────────────────────────
#IN THIS ITERATION, IT FIXES THE RATE-LIMIT FOR THE API. A BACKUP OUTPUT IS PROVIDED FOR CONSISTENCY

def ask_ai(messages):
    """
    Calls the model. Raises RuntimeError("RATE_LIMIT_429") on rate-limit.
    Fix (Iteration 3): Isolated 429 error so we can apply a local fallback.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=512,
            messages=messages,
            temperature=0.2,   # Low temp = consistent, factual responses
        )
        return response.choices[0].message.content
    except Exception as e:
        msg = str(e)
        if "Error code: 429" in msg or "Rate limit exceeded" in msg:
            raise RuntimeError("RATE_LIMIT_429")
        raise


# ─── JSON Safety + Tool Dispatch ─────────────────────────────────────────────
def safe_json_loads(s: str) -> dict:
    """
    Fix (Iteration 2→3): Model sometimes wraps JSON in markdown fences.
    This strips fences and extracts the first JSON object found.
    """
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        # Strip markdown fences if present
        cleaned = s.strip().strip("```json").strip("```").strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass
        # Last resort: find first {...} block
        m = re.search(r"\{.*\}", s, re.S)
        if m:
            return json.loads(m.group(0))
        raise


def tool_dispatch(action_name: str, action_input: dict) -> dict:
    """
    STAGE 3 — GENERATE
    Routes the AI's chosen action to the correct Python function.
    Returns a dict observation that gets fed back to the AI as STAGE 4.
    """
    if action_name == "search_policies":
        return search_policies(action_input.get("topic", ""))
    if action_name == "check_order":
        return check_order(action_input.get("order_number", ""))
    if action_name == "compute_return_eligibility":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for eligibility check."}
        return compute_return_eligibility(order)
    if action_name == "generate_return_instructions":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for return instructions."}
        return generate_return_instructions(
            order=order,
            preference=action_input.get("preference", ""),
            reason=action_input.get("reason", ""),
        )
    if action_name == "escalate_to_human":
        return escalate_to_human(action_input.get("reason", ""))
    return {"error": f"Unknown action: {action_name}"}


def local_refund_policy_answer() -> str:
    """
    Fix (Iteration 3): Local fallback when API is rate-limited and the user
    is asking about refund/return policy. Uses POLICY_DB directly.
    """
    p = POLICY_DB["returns_policy"]
    options = POLICY_DB["return_options"]
    return (
        f"Our return window is {p['window_days']} days from delivery "
        f"(items should be {p['condition']}). "
        f"Refunds are issued to your {p['refund_method']} within {p['refund_timeline_days']}. "
        f"Return options include: {options[0]['name']}, {options[1]['name']}, "
        f"or {options[2]['name']}. "
        "If you share your order number, I can confirm your eligibility and next steps."
    )


# ─── ReACT Agent Loop ────────────────────────────────────────────────────────
#PROCEDURE FOR THE AI TO HANDLE CUSTOMER SUPPORT, BASED ON A SUBSET THAT USES 6 STAGES TO BEHAVE DIFFERENTLY.
#STAGE 1 IS NORMALIZED, ALL THE WAY UP TO STAGE 5, WHERE IT BECOMES EXTREMELY CRITICAL OF PROVIDING A PRECISE RESPONSE BEFORE ENDING CONVERSATION
#THIS IS TO ENSURE SAFEGUARDS

def run_react_agent(user_summary: str, followup_messages: list, logger: ConversationLogger) -> str:
    """
    Implements the full ReACT loop:
      Stage 1: AI receives user summary → reasons internally
      Stage 2: AI emits {"type": "action", "name": ..., "input": ...}
      Stage 3: tool_dispatch() runs the tool
      Stage 4: Observation appended as user message
      Stage 5: Loop continues until type=final or max_steps reached
    """
    messages = [{"role": "system", "content": REACT_SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": user_summary})
    for m in followup_messages:
        messages.append(m)

    max_steps = 6   # Guard against infinite tool loops

    for step in range(max_steps):
        # ── STAGE 1/2: AI reasons and plans ──────────────────────────────────
        try:
            raw = ask_ai(messages)
        except RuntimeError as e:
            if str(e) == "RATE_LIMIT_429":
                # Check if last user message was about refund policy
                last_user = next(
                    (m.get("content", "") for m in reversed(followup_messages) if m.get("role") == "user"),
                    ""
                )
                if re.search(r"\b(refund|return|policy)\b", last_user.lower()):
                    return local_refund_policy_answer()
                return (
                    "I'm temporarily rate-limited. Please try again in a minute, "
                    "or get a free token at https://token.llm7.io/. "
                    f"You can also email {COMPANY_CONFIG['support_email']} for help within 1 business day."
                )
            raise

        logger.log("assistant_raw", raw)

        # ── STAGE 3: Parse AI output ──────────────────────────────────────────
        try:
            obj = safe_json_loads(raw)
        except Exception:
            # Parsing failed entirely — ask for clarification
            return (
                "I had trouble reading the details. "
                "Could you confirm your order number and whether you prefer a refund or exchange?"
            )

        # ── STAGE 5: Check if done ────────────────────────────────────────────
        if obj.get("type") == "final":
            return obj.get("message", "How can I help?")

        # ── STAGE 2→3: Dispatch tool ──────────────────────────────────────────
        if obj.get("type") == "action":
            name = obj.get("name", "")
            action_input = obj.get("input", {}) or {}
            observation = tool_dispatch(name, action_input)

            logger.log("tool_call", {"step": step + 1, "name": name, "input": action_input})
            logger.log("tool_observation", observation)

            # ── STAGE 4: Feed observation back ───────────────────────────────
            messages.append({
                "role": "assistant",
                "content": json.dumps({"type": "action", "name": name, "input": action_input})
            })
            messages.append({
                "role": "user",
                "content": json.dumps({"type": "observation", "name": name, "output": observation})
            })
            continue

        # ── STAGE 5 (FIX): Schema error feedback ─────────────────────────────
        # Fix (Iteration 2): AI sometimes outputs plain text; redirect it.
        messages.append({
            "role": "user",
            "content": json.dumps({
                "type": "observation",
                "name": "schema_error",
                "output": {"error": "You must output JSON with type=action or type=final. No prose."}
            })
        })

    # Max steps exceeded → escalate
    return (
        "I'm going to connect you with a human agent to finish this quickly. "
        f"Please email {COMPANY_CONFIG['support_email']} with your order number and request."
    )


# ─── Main Support Flow ────────────────────────────────────────────────────────
#FORMATS THE MAIN MENU, PROMPTING THE CUSTOMER TO MANUALLY TYPE IN AN INPUT

def run_support_session(logger):
    show_main_menu()

    while True:
        choice = input("  Enter number (1-6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print(f"\n  You selected: {category}")
            break
        print("  Please enter a number between 1 and 6.")

    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Connecting you with our support agent...\n")
    print_divider()

    followups = []
    agent_reply = run_react_agent(summary, followups, logger)
    logger.log("assistant", agent_reply)

    print("Support Agent:\n")
    print("  " + agent_reply + "\n")
    print_divider()

    print("  Need anything else? Type a follow-up question or 'done' to exit.\n")

    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue
        if follow_up.lower() in ("done", "quit", "exit"):
            print("\n  Agent: Thanks for contacting us - have a great day!\n")
            break
        followups.append({"role": "user", "content": follow_up})
        logger.log("user_followup", follow_up)
        reply = run_react_agent(summary, followups, logger)
        followups.append({"role": "assistant", "content": reply})
        logger.log("assistant_followup", reply)
        print(f"\n  Agent: {reply}\n")


# ─── Entry Point ─────────────────────────────────────────────────────────────
#ENDING CUTOFF FOR CUSTOMER SUPPORT TICKETS

def main():
    print("=" * 60)
    print(f"  {COMPANY_CONFIG['name']} - Customer Support (ReACT Demo)")
    print(f"  {COMPANY_CONFIG['support_hours']}")
    print(f"  {COMPANY_CONFIG['support_email']}")
    print("=" * 60)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except KeyboardInterrupt:
            print("\n\n  Session interrupted.")
        except Exception as e:
            print(f"\n  Something went wrong: {e}")
            print("  The session will now close safely.")
        finally:
            logger.save()

        again = input("\n  Start a new support session? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print(f"\n  Thank you for contacting {COMPANY_CONFIG['name']} support. Have a great day!\n")
            break

main()

  Acme Corp - Customer Support (ReACT Demo)
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
------------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
------------------------------------------------------------
  Enter number (1-6): 2

  You selected: Returns & Refunds

  Let's gather some details about your Returns & Refunds issue.

  What is your order number? (e.g. ORD-12345 or 12345) 12345
  Why are you returning?
   1. Item is defective
   2. Wrong item received
   3. Changed my mind
   4. Item not as described
   Enter 1-4:  1
  Would you prefer a refund or exchange? (type refund or exchange):  refund
  What email is on your account? james.h.phan@sjsu.edu

  Connecting you with our support agent...

------------------------------------------------------------
Support Agent:

  Thank you for 